# Assessment 1 — Analysing historical data with system performance

**Student ID:** 33602190

## 1. Spark setup

### 1.1 Create the Spark session

In [1]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

master = "local[*]"
app_name = "Assessment1-AML"
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 08:53:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/10/03 08:53:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### 1.2 Execution environment

In [2]:
print("Execution mode:", sc.master)
print("Processing threads:", sc.defaultParallelism)
print("Driver memory:", sc.getConf().get("spark.driver.memory"))
print("Executor memory: shared driver JVM in local mode; no separate executor heap")

Execution mode: local[*]
Processing threads: 14
Driver memory: 4g
Executor memory: shared driver JVM in local mode; no separate executor heap


## 2. Load and validate the data

The two HI-Small CSV files are loaded in full for validation. The transaction schema assigns unique names to the sender and receiver columns in their original CSV order. Account identifiers remain strings; bank identifiers are numeric in both datasets. Malformed records cause loading to fail rather than being silently discarded. No rows are removed or sampled here.

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, TimestampType, LongType,
    StringType, DoubleType, IntegerType
)

transaction_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("from_bank", LongType(), True),
    StructField("sender_account", StringType(), True),
    StructField("to_bank", LongType(), True),
    StructField("receiver_account", StringType(), True),
    StructField("amount_received", DoubleType(), True),
    StructField("receiving_currency", StringType(), True),
    StructField("amount_paid", DoubleType(), True),
    StructField("payment_currency", StringType(), True),
    StructField("payment_format", StringType(), True),
    StructField("is_laundering", IntegerType(), True)
])

account_schema = StructType([
    StructField("bank_name", StringType(), True),
    StructField("bank_id", LongType(), True),
    StructField("account_number", StringType(), True),
    StructField("entity_id", StringType(), True),
    StructField("entity_name", StringType(), True)
])

transactions = (
    spark.read.schema(transaction_schema)
    .option("header", True)
    .option("enforceSchema", True)
    .option("timestampFormat", "yyyy/MM/dd HH:mm")
    .option("mode", "FAILFAST")
    .csv("data/HI-Small_Trans.csv")
)
accounts = (
    spark.read.schema(account_schema)
    .option("header", True)
    .option("enforceSchema", True)
    .option("mode", "FAILFAST")
    .csv("data/HI-Small_accounts.csv")
)

transaction_count = transactions.count()
account_count = accounts.count()
print(f"Transactions: {transaction_count:,} rows; {len(transactions.columns)} columns")
print(f"Accounts: {account_count:,} rows; {len(accounts.columns)} columns")
transactions.printSchema()
accounts.printSchema()

Transactions: 5,078,345 rows; 11 columns
Accounts: 518,581 rows; 5 columns
root
 |-- timestamp: timestamp (nullable = true)
 |-- from_bank: long (nullable = true)
 |-- sender_account: string (nullable = true)
 |-- to_bank: long (nullable = true)
 |-- receiver_account: string (nullable = true)
 |-- amount_received: double (nullable = true)
 |-- receiving_currency: string (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- payment_currency: string (nullable = true)
 |-- payment_format: string (nullable = true)
 |-- is_laundering: integer (nullable = true)

root
 |-- bank_name: string (nullable = true)
 |-- bank_id: long (nullable = true)
 |-- account_number: string (nullable = true)
 |-- entity_id: string (nullable = true)
 |-- entity_name: string (nullable = true)



### 2.1 Missing values and valid values

Check every field for nulls and string fields for blanks. Check that laundering labels are 0 or 1 and monetary amounts are finite and non-negative. Amounts retain the proposed double type; monetary totals must be grouped by currency, and later result comparisons must allow for floating-point precision.

In [4]:
missing_results = []
for dataset_name, dataframe in [("Transactions", transactions), ("Accounts", accounts)]:
    checks = []
    for field in dataframe.schema.fields:
        missing = F.col(field.name).isNull()
        if isinstance(field.dataType, StringType):
            missing = missing | (F.trim(F.col(field.name)) == "")
        checks.append(F.sum(F.when(missing, 1).otherwise(0)).alias(field.name))
    counts = dataframe.agg(*checks).first().asDict()
    missing_results.extend((dataset_name, column, count) for column, count in counts.items())

spark.createDataFrame(missing_results, ["dataset", "column", "missing_rows"]).show(16, truncate=False)

transactions.agg(
    F.sum(F.when(~F.col("is_laundering").isin(0, 1), 1).otherwise(0)).alias("invalid_labels"),
    *[
        F.sum(F.when(
            F.isnan(column) | (F.abs(F.col(column)) == float("inf")) | (F.col(column) < 0),
            1
        ).otherwise(0)).alias(f"invalid_{column}")
        for column in ["amount_paid", "amount_received"]
    ]
).show(truncate=False)

+------------+------------------+------------+
|dataset     |column            |missing_rows|
+------------+------------------+------------+
|Transactions|timestamp         |0           |
|Transactions|from_bank         |0           |
|Transactions|sender_account    |0           |
|Transactions|to_bank           |0           |
|Transactions|receiver_account  |0           |
|Transactions|amount_received   |0           |
|Transactions|receiving_currency|0           |
|Transactions|amount_paid       |0           |
|Transactions|payment_currency  |0           |
|Transactions|payment_format    |0           |
|Transactions|is_laundering     |0           |
|Accounts    |bank_name         |0           |
|Accounts    |bank_id           |0           |
|Accounts    |account_number    |0           |
|Accounts    |entity_id         |0           |
|Accounts    |entity_name       |0           |
+------------+------------------+------------+



+--------------+-------------------+-----------------------+
|invalid_labels|invalid_amount_paid|invalid_amount_received|
+--------------+-------------------+-----------------------+
|0             |0                  |0                      |
+--------------+-------------------+-----------------------+



### 2.2 Date coverage, labels and sender cardinality

The source timestamps contain no timezone offset; Spark interprets them using the session timezone shown below. Label counts describe simulated laundering already identified in the source, rather than predictions.

In [5]:
print("Timestamp parsing timezone:", spark.conf.get("spark.sql.session.timeZone"))
transactions.agg(
    F.min("timestamp").alias("first_transaction"),
    F.max("timestamp").alias("last_transaction"),
    F.countDistinct("sender_account").alias("distinct_sender_accounts")
).show(truncate=False)

transactions.groupBy("is_laundering").count().withColumn(
    "percentage", F.round(F.col("count") / transaction_count * 100, 4)
).orderBy("is_laundering").show()

Timestamp parsing timezone: Australia/Melbourne


+-------------------+-------------------+------------------------+
|first_transaction  |last_transaction   |distinct_sender_accounts|
+-------------------+-------------------+------------------------+
|2022-09-01 00:00:00|2022-09-18 16:18:00|496995                  |
+-------------------+-------------------+------------------------+



+-------------+-------+----------+
|is_laundering|  count|percentage|
+-------------+-------+----------+
|            0|5073168|   99.8981|
|            1|   5177|    0.1019|
+-------------+-------+----------+



### 2.3 Validate account joins

Check uniqueness of the composite account key `(bank_id, account_number)` and count transaction rows with unmatched sender or receiver keys. Duplicate account keys could multiply transactions when joined; unmatched keys could cause an inner join to drop transactions.

In [6]:
duplicate_account_keys = (
    accounts.groupBy("bank_id", "account_number")
    .count().filter(F.col("count") > 1).count()
)
account_keys = accounts.select("bank_id", "account_number").distinct()

unmatched_senders = transactions.join(
    account_keys,
    (transactions.from_bank == account_keys.bank_id)
    & (transactions.sender_account == account_keys.account_number),
    "left_anti"
).count()
unmatched_receivers = transactions.join(
    account_keys,
    (transactions.to_bank == account_keys.bank_id)
    & (transactions.receiver_account == account_keys.account_number),
    "left_anti"
).count()

spark.createDataFrame([
    ("Duplicate account key groups", duplicate_account_keys),
    ("Transactions with unmatched sender", unmatched_senders),
    ("Transactions with unmatched receiver", unmatched_receivers)
], ["check", "count"]).show(truncate=False)

+------------------------------------+-----+
|check                               |count|
+------------------------------------+-----+
|Duplicate account key groups        |0    |
|Transactions with unmatched sender  |0    |
|Transactions with unmatched receiver|0    |
+------------------------------------+-----+



The files contain **5,078,345 transactions** and **518,581 account records**. No null or blank fields, invalid laundering labels, or negative/non-finite amounts were found. Transaction timestamps run from **1 September 2022 00:00** to **18 September 2022 16:18**.

There are **496,995 distinct sender account identifiers**. **5,177 transactions (0.1019%)** are labelled as laundering, so the final query threshold must account for the small labelled population.

The composite account key is unique and all sender/receiver keys match. Joining on both bank and account identifiers can therefore preserve transaction counts. No data cleaning or row removal has been applied.